# **🛰️ Game 2 — The Data Reconnaissance Mission**
## Map the Battlefield Before You Train

Your team has received a new version of the dataset.

At first glance, the data looks ready for modeling.  
But real-world datasets often hide risks that can silently reduce performance, mislead evaluation, or create unstable systems.

Your mission is to investigate the dataset before training any model.

Do not fix the data yet.

First, discover what is happening.

You must inspect the images, texts, labels, and differences between Train and Validation. Then build an evidence-based report that explains which risks should be handled first.

### Final Question

> What can silently mislead your model before training begins?

In [ ]:
# ============================================================
# Cell 1 — Setup
# ============================================================
from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import re

drive.mount("/content/drive")

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

ROOT = Path("/content/drive/MyDrive/AI_Olympics_2026")

STUDENT_PACKAGE_DIR = ROOT / "05_student_package"

GAME2_DIR = (
    STUDENT_PACKAGE_DIR
    / "games"
    / "game2_data_reconnaissance"
)

print("✅ Setup completed")

In [ ]:
# ============================================================
# Cell 2 — Load Game 2 Data
# ============================================================
train_df = pd.read_csv(
    GAME2_DIR / "game2_train_eda.csv"
)

validation_df = pd.read_csv(
    GAME2_DIR / "game2_validation_eda.csv"
)

print("✅ Game 2 data loaded")

print(f"Train samples      : {len(train_df):,}")
print(f"Validation samples : {len(validation_df):,}")

display(train_df.head())
display(validation_df.head())

In [ ]:
# ============================================================
# Cell 3 — Resolve Image Paths
# ============================================================
def resolve_image_path(relative_path):
    relative_path = str(relative_path)

    candidate_paths = [
        STUDENT_PACKAGE_DIR / relative_path,
        ROOT / relative_path,
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Image not found: {relative_path}"
    )


example_path = resolve_image_path(
    train_df.iloc[0]["image_path"]
)

print("✅ Example image found:")
print(example_path)

In [ ]:
# ============================================================
# Cell 4 — Initial Dataset Overview
# ============================================================
print("Train shape:")
print(train_df.shape)

print("\nValidation shape:")
print(validation_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nTrain missing values:")
display(
    train_df.isna()
    .sum()
    .rename("missing_values")
    .reset_index()
)

print("\nValidation missing values:")
display(
    validation_df.isna()
    .sum()
    .rename("missing_values")
    .reset_index()
)

In [ ]:
# ============================================================
# Cell 5 — TODO: Analyze Label Distribution
# ============================================================
# Your task:
# 1. Count the number of samples per class.
# 2. Compare Train and Validation.
# 3. Visualize the distributions.
# 4. Explain whether the dataset is balanced.
#
# Write your code below.

In [ ]:
# ============================================================
# Cell 6 — TODO: Analyze Text Quality
# ============================================================
# Your task:
# 1. Inspect missing or empty texts.
# 2. Calculate text length in characters and words.
# 3. Look for unusual text patterns.
# 4. Compare Train and Validation.
# 5. Show examples of suspicious samples.
#
# Write your code below.

In [ ]:
# ============================================================
# Cell 7 — Starter Text Statistics
# ============================================================
def safe_text(value):
    if pd.isna(value):
        return ""

    return str(value)


def build_text_statistics(dataframe):
    output = dataframe.copy()

    output["safe_text"] = output["text"].map(
        safe_text
    )

    output["text_length_chars"] = (
        output["safe_text"]
        .str.len()
    )

    output["text_length_words"] = (
        output["safe_text"]
        .str.split()
        .str.len()
    )

    output["is_empty_text"] = (
        output["safe_text"]
        .str.strip()
        .eq("")
    )

    output["symbol_count"] = (
        output["safe_text"]
        .str.count(r"[^A-Za-z0-9\s]")
    )

    output["symbol_ratio"] = (
        output["symbol_count"]
        / output["text_length_chars"]
        .clip(lower=1)
    )

    return output


train_text_stats = build_text_statistics(
    train_df
)

validation_text_stats = build_text_statistics(
    validation_df
)

print("✅ Starter text statistics created")

In [ ]:
# ============================================================
# Cell 8 — TODO: Explore Text Statistics
# ============================================================
# Your task:
# Use train_text_stats and validation_text_stats.
#
# Recommended directions:
# - summary statistics
# - histograms
# - boxplots
# - empty text counts
# - shortest texts
# - longest texts
# - unusual symbol ratios
#
# Write your code below.

In [ ]:
# ============================================================
# Cell 9 — Image Statistics Helper
# ============================================================
!pip -q install opencv-python-headless tqdm

import cv2

from PIL import Image
from tqdm.auto import tqdm


def extract_image_statistics(dataframe):
    records = []

    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc="Analyzing images"
    ):
        image_path = resolve_image_path(
            row["image_path"]
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        image_array = np.asarray(image)

        gray = cv2.cvtColor(
            image_array,
            cv2.COLOR_RGB2GRAY
        )

        width, height = image.size

        records.append({
            "sample_id": row["sample_id"],
            "image_path": row["image_path"],
            "label": row["label"],
            "width": width,
            "height": height,
            "aspect_ratio": width / max(height, 1),
            "brightness_mean": float(
                gray.mean()
            ),
            "contrast_std": float(
                gray.std()
            ),
            "laplacian_variance": float(
                cv2.Laplacian(
                    gray,
                    cv2.CV_64F
                ).var()
            ),
        })

    return pd.DataFrame(records)

In [ ]:
# ============================================================
# Cell 10 — Extract Image Statistics
# ============================================================

# can use like this cell or create own cell
'''train_image_stats = extract_image_statistics(
    train_df
)

validation_image_stats = extract_image_statistics(
    validation_df
)

print("✅ Image statistics created")

display(train_image_stats.head())'''

In [ ]:
# ============================================================
# Cell 11 — TODO: Analyze Image Quality
# ============================================================
# Your task:
# Use train_image_stats and validation_image_stats.
#
# Investigate:
# - image dimensions
# - aspect ratios
# - brightness distributions
# - contrast distributions
# - visual sharpness indicators
# - unusual samples
#
# Create:
# - summary tables
# - visualizations
# - suspicious-sample tables
#
# Write your code below.

In [ ]:
# ============================================================
# Cell 12 — Visualize Selected Images
# ============================================================
from PIL import Image


def show_samples(
    dataframe,
    title,
    n=8,
):
    preview_rows = dataframe.head(n)

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(16, 8)
    )

    axes = axes.flatten()

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(
        axes,
        preview_rows.iterrows()
    ):
        source_row = train_df[
            train_df["sample_id"]
            == row["sample_id"]
        ]

        if len(source_row) == 0:
            source_row = validation_df[
                validation_df["sample_id"]
                == row["sample_id"]
            ]

        image_path = source_row.iloc[0][
            "image_path"
        ]

        image = Image.open(
            resolve_image_path(
                image_path
            )
        ).convert("RGB")

        ax.imshow(image)

        ax.set_title(
            f"{row['sample_id']}\n"
            f"Label: {row['label']}",
            fontsize=9
        )

    plt.suptitle(
        title,
        fontsize=14
    )

    plt.tight_layout()
    plt.show()


print("✅ Visualization helper ready")

In [ ]:
# ============================================================
# Cell 13 — TODO: Build Visual Evidence
# ============================================================
# Your task:
# 1. Select suspicious image samples from your analysis.
# 2. Visualize them.
# 3. Explain why they should be investigated.
#
# Example:
#
# suspicious_images = train_image_stats.sort_values(
#     "laplacian_variance"
# ).head(8)
#
# show_samples(
#     suspicious_images,
#     title="Selected Samples"
# )

In [ ]:
# ============================================================
# Cell 14 — TODO: Compare Train and Validation
# ============================================================
# Your task:
# Compare Train and Validation across:
#
# - label distribution
# - missing values
# - text lengths
# - image dimensions
# - brightness
# - contrast
# - sharpness indicators
#
# Identify any differences that may affect model evaluation.
#
# Write your code below.

In [ ]:
# ============================================================
# Cell 15 — TODO: Build Risk Priority Table
# ============================================================
# Your task:
# Create a table that summarizes the risks you discovered.
#
# Required columns:
#
# risk_id
# modality
# discovered_pattern
# evidence
# affected_split
# affected_samples
# severity
# recommended_next_action
#
# Example:
#
# risk_table = pd.DataFrame([
#     {
#         "risk_id": "R01",
#         "modality": "...",
#         "discovered_pattern": "...",
#         "evidence": "...",
#         "affected_split": "...",
#         "affected_samples": "...",
#         "severity": "...",
#         "recommended_next_action": "...",
#     }
# ])
#
# display(risk_table)

In [ ]:
# ============================================================
# Cell 16 — Save Required Outputs
# ============================================================
TEAM_NAME = "replace_with_team_name"

# Can save like this or create own way
'''OUTPUT_DIR = (
    ROOT
    / "Game2_Submissions"
    / TEAM_NAME
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

train_text_stats.to_csv(
    OUTPUT_DIR / "game2_train_text_statistics.csv",
    index=False
)

validation_text_stats.to_csv(
    OUTPUT_DIR / "game2_validation_text_statistics.csv",
    index=False
)

train_image_stats.to_csv(
    OUTPUT_DIR / "game2_train_image_statistics.csv",
    index=False
)

validation_image_stats.to_csv(
    OUTPUT_DIR / "game2_validation_image_statistics.csv",
    index=False
)

# Save this after creating risk_table:
#
# risk_table.to_csv(
#     OUTPUT_DIR / "game2_risk_priority_table.csv",
#     index=False
# )

print("✅ Game 2 outputs saved")
print(f"📁 {OUTPUT_DIR}")'''

# **Final Conclusion**

Write your findings clearly.

Your conclusion must answer:

1. Is the dataset ready for modeling?
2. What risks did your team discover?
3. Which risks affect images?
4. Which risks affect texts?
5. Are Train and Validation fully comparable?
6. Which issues should be handled first?
7. What preprocessing strategies should be tested in the next game?

Do not fix the dataset in this game.

Your task is to discover, measure, and prioritize.



---



**Game2_Submission_TeamName/**

├── Game_2_Data_Reconnaissance_Mission_Completed.ipynb

├── game2_train_text_statistics.csv

├── game2_validation_text_statistics.csv

├── game2_train_image_statistics.csv

├── game2_validation_image_statistics.csv

└── game2_risk_priority_table.csv



---



**game2_risk_priority_table.csv:**


risk_id,modality,discovered_pattern,evidence,affected_split,affected_samples,severity,recommended_next_action



---



**Game_2_Data_Reconnaissance_Mission** --> 100 point